In [24]:
import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
import qiskit
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [25]:
atom: str = "H"
natoms: int = 6

In [26]:
def generate_linear_geometry(atom: str, natoms: int, atomic_distance: float = 1.0) -> str:
    """Returns a linear Hydrogen chain geometry for use in PySCF molecule construction.
    
    Args:
        natoms: Number of Hydrogen atoms in the chain.
        atomic_distance: Equal spacing between Hydrogen atoms.
    """
    return "; ".join([f"{atom} 0 0 {i * atomic_distance}" for i in range(natoms)])

In [27]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=generate_linear_geometry(atom, natoms),
    basis="sto-6g",
)

# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
# reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -3.15600092954731
norb = 6
nelec = (3, 3)


In [28]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -3.257214530748513  E_corr = -0.101213601201206


In [29]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_grid(
    num_rows=int(np.ceil(np.sqrt(2 * norb))),
    num_columns=int(np.ceil(np.sqrt(2 * norb)))
)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
try:
    pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
        backend=backend,
        norb=norb,
        connectivity="heavy-hex",
        interaction_pairs=(pairs_aa, pairs_ab),
        optimization_level=3,
    )
    print("Unable to generate ffsim pass manager")
except RuntimeError:
    pass_manager = None

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
# circuit.measure_all()

Unable to generate ffsim pass manager


In [30]:
if pass_manager is not None:
    compiled = pass_manager.run(circuit)
else:
    compiled = qiskit.transpile(circuit, backend=backend)

In [31]:
print(f"Number of qubits: {compiled.num_qubits}")
print(f"Gate counts: {compiled.count_ops()}")

Number of qubits: 16
Gate counts: OrderedDict({'xx_plus_yy': 48, 'cp': 12, 'p': 12, 'x': 6, 'swap': 4})


In [32]:
compiled.draw(fold=-1)

┌───┐                            ┌─────────────────────────┐  ┌──────────────────────────┐                             ┌───────────────────────────┐                                                                                                                                                                                                     ┌──────────────────────────┐┌──────────────────────────┐┌──────────────────────────┐                                                                                       ┌──────────────────────────┐┌──────────────────────────┐                                                        ┌─────────────────────────┐         ┌────────────┐                                                 
      q_1 -> 2 ┤ X ├────────────────────────────┤1                        ├──┤0                         ├─────────────────────────────┤1                          ├─────────■──────────────────────────────────────────────────────────────────────────────────────────────────■────────────────────────────────────────────────────────────────────────────────────────┤0                         ├┤1                         ├┤0                         ├───────────────────────────────────────────────────────────────────────────────────────┤1                         ├┤0                         ├────────────────────────────────────────────────────────┤0                        ├─────────┤ P(-1.6647) ├─────────────────────────────────────────────────
               ├───┤                            │                         │  │  (XX+YY)(2.9884,-2.3664) │                             │                           │         │P(-0.73506)                                                                                       │                                                                                        │  (XX+YY)(3.1311,-1.5768) ││                          ││  (XX+YY)(3.1161,-3.1732) │                                                                                       │                          ││  (XX+YY)(1.8375,-3.1483) │      ┌─────────────┐                                   │                         │         └────────────┘                                                 
      q_0 -> 3 ┤ X ├────────────────────────────┤                         ├──┤1                         ├─────────────────────────────┤                           ├─────────■────────────────────────────────────────────────────────────────────X─────────────────────────────┼─────────────────────────────────────────────────────────────────────────X──────────────┤1                         ├┤                          ├┤1                         ├───────────────────────────────────────────────────────────────────────────────────────┤                          ├┤1                         ├──────┤ P(-0.87732) ├───────────────────────────────────┤                         ├────────────────────────────────────────────────────────────────────────
               └───┘                            │  (XX+YY)(3.1416,1.3429) │  ├──────────────────────────┤                             │  (XX+YY)(3.1416,-0.64046) │                                                                              │                             │                                                                         │              └──────────────────────────┘│  (XX+YY)(2.9884,-4.5721) │├─────────────────────────┬┘                               ┌──────────────────────────┐                            │  (XX+YY)(3.1264,-4.5984) │└──────────────────────────┘      └─────────────┘      ┌───────────────────────────┐│  (XX+YY)(3.1153,1.5132) │         ┌────────────┐                                                 
      q_5 -> 5 ─────────────────────────────────┤                         ├──┤0                         ├─────────────────────────────┤                           ├──────────────────────────────■───────────────────────────────────────────────┼─────────────────────────────┼────────────────────────────

In [33]:
from qiskit.quantum_info import Statevector, SparsePauliOp

In [34]:
statevector = Statevector(compiled)
print(statevector)

observable = SparsePauliOp("ZZZZ")
sv_expectation_value = statevector.expectation_value(observable).real
print(sv_expectation_value)

Statevector([0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
            dims=(2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2))
0.6633345860002099


In [35]:
from propaq.datatypes._abstract import BitMask

from propaq.datatypes.majorana import MajoranaMonomial

from propaq.propagators import MajoranaPropagator
from propaq.circuits import MajoranaCircuit 
from propaq.noise import UniformNoiseModel, truncation
from propaq.noise import TruncationPolicy 

from propaq.datatypes import MajoranaTermSum

In [36]:
mc = MajoranaCircuit.from_qiskit(compiled.copy(), n_modes=2 * compiled.num_qubits)

In [37]:
observable_mts = MajoranaTermSum.from_sparse_pauli_op(observable)
observable_mts.items()[0][0].modes

255

In [38]:
truncation_policy = TruncationPolicy(weight_cutoff=100000, coeff_cutoff=1e-16)

In [39]:
prop = MajoranaPropagator(None, truncation_policy, n_threads=1)

In [40]:
mp_expectation_value = prop.expectation_value(observable_mts, mc, fock_state=0)
mp_expectation_value

0.6633345860002114